# CreditSense — EDA

This notebook feeds **Section 1 of the report** (Data Exploration & Preprocessing, 30%).

Run after `credit_train.csv` and `credit_test.csv` are placed in `../data/`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import load_data, TARGET_A, TARGET_B

sns.set_theme(style='whitegrid', context='notebook')
FIG = pathlib.Path('../report/figures')
FIG.mkdir(parents=True, exist_ok=True)

train, test = load_data()
print('train:', train.shape, '  test:', test.shape)
train.head()

## 1. Target distributions

Two targets: `RiskTier` (classification) and `InterestRate` (regression). We need to know whether classes are balanced (affects loss choice) and whether rate is bimodal/clustered by tier (affects stacking strategy).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train[TARGET_A].value_counts().sort_index().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('RiskTier distribution (Task A)')
axes[0].set_xlabel('RiskTier'); axes[0].set_ylabel('Count')

axes[1].hist(train[TARGET_B], bins=60, color='indianred')
axes[1].set_title('InterestRate distribution (Task B)')
axes[1].set_xlabel('APR (%)')

sns.boxplot(data=train, x=TARGET_A, y=TARGET_B, ax=axes[2])
axes[2].set_title('InterestRate by RiskTier (monotone \u2192 cross-task signal)')
plt.tight_layout()
plt.savefig(FIG/'targets.png', dpi=140, bbox_inches='tight')
plt.show()

## 2. Missing-value structure

The PDF tells us missingness is **structural** (e.g., `PropertyValue` missing ⇔ renter). Confirming this visually justifies our strategy: keep `*_was_missing` indicators instead of pure imputation.

In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f'{len(miss)} columns have missing values')
fig, ax = plt.subplots(figsize=(10, max(4, 0.3*len(miss))))
miss.plot.barh(ax=ax, color='darkorange')
ax.set_xlabel('Fraction missing')
ax.set_title('Missingness by column (train)')
plt.tight_layout()
plt.savefig(FIG/'missingness.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# If HomeOwnership + PropertyValue both exist, verify the structural hypothesis
if 'HomeOwnership' in train.columns and 'PropertyValue' in train.columns:
    tbl = pd.crosstab(train['HomeOwnership'], train['PropertyValue'].isna(), 
                      normalize='index').round(3)
    tbl.columns = ['has_value', 'missing']
    display(tbl)

## 3. Correlation with targets

Identifies the raw predictors most correlated with each target — informs which engineered features to prioritise.

In [ ]:
numeric = train.select_dtypes(include=np.number).drop(columns=['Id'], errors='ignore')
corr_a = numeric.corr()[TARGET_A].drop([TARGET_A, TARGET_B]).abs().sort_values(ascending=False)
corr_b = numeric.corr()[TARGET_B].drop([TARGET_A, TARGET_B]).abs().sort_values(ascending=False)

top = pd.concat([corr_a.head(15).rename('|corr(RiskTier)|'),
                 corr_b.head(15).rename('|corr(InterestRate)|')], axis=1).fillna(0)
display(top)

fig, ax = plt.subplots(figsize=(8, 6))
top.plot.barh(ax=ax)
ax.set_title('Top-15 absolute correlations with each target')
plt.tight_layout()
plt.savefig(FIG/'correlations.png', dpi=140, bbox_inches='tight')
plt.show()

## 4. Skew check (log-transform candidates)

Money-denominated features (incomes, balances) are right-skewed log-normal. We'll log1p-transform these in `features.py`.

In [ ]:
money_cols = [c for c in numeric.columns if any(k in c.lower() 
              for k in ('income','amount','balance','value'))][:6]
if money_cols:
    fig, axes = plt.subplots(2, len(money_cols), figsize=(3*len(money_cols), 6))
    for i, c in enumerate(money_cols):
        train[c].dropna().plot.hist(ax=axes[0][i], bins=40)
        axes[0][i].set_title(f'{c}\n(raw)')
        np.log1p(train[c].clip(lower=0).dropna()).plot.hist(ax=axes[1][i], bins=40, color='g')
        axes[1][i].set_title(f'log1p({c})')
    plt.tight_layout()
    plt.savefig(FIG/'skew.png', dpi=140, bbox_inches='tight')
    plt.show()

## 5. Categorical cardinality & target interaction

In [ ]:
cats = train.select_dtypes(exclude=np.number).columns.tolist()
for c in cats:
    print(f'{c}: {train[c].nunique()} unique, top: {train[c].value_counts().head(3).to_dict()}')


## Takeaways (put into report Section 1)

- Class balance of `RiskTier`: [fill from output].
- Strongest raw predictors of each task: [fill from top-15 correlation table].
- Missingness is structural in [list columns] — justifies keeping `*_was_missing` flags.
- Money-like columns are heavily right-skewed; log1p is appropriate.
- `InterestRate` vs `RiskTier` box plot shows monotone relation → cross-task stacking will pay off.